## Tarea 3: ¿Cómo vamos a entrenar la RN?



Instrucciones: 
1. Ponle de nombre a tu tarea `tarea_3.ipynb` un Jupyter Notebook (5 points) 
2. Escribe tu nombre comleto y tus iniciales en tu notebook después de la siguiente nota: 
*Este trabajo es mio y sigue la integridad académica del Tec. de Monterrey*
3. Agrega los nombres de todos los compañeros con los que trabajaste. Recuerda que puedes discutir con tus compañeros pero cada quien tiene que entregar su tarea.
4. **ADVERTENCIA!** no se permite el uso de LLMs para estas tareas! El aprendizaje profundo no va a hacer la tarea de aprendizaje profundo... Si no entienden bien los conceptos no hay diferencia entre ustedes y el LLM y esto es sad, se mueren las hadas cuando hacen esto :(
5. Asegurate que tu código corra!
6. Las preguntas de teoria contestalas en texto usando markdown, no comentarios en python

1. Considere construir un modelo para predecir el número de peatones $y\in \{0, 1, 2, . . .\}$ que pasarán por un punto dado de la ciudad en el próximo minuto, basándose en los datos x que contienen información sobre la hora del día, la longitud y la latitud, y el tipo de barrio.
Una distribución adecuada para modelar conteos es la distribución de Poisson, esta tiene un único parámetro $\lambda >0$ llamado tasa que representa la media de la distribución. La distribución es la siguiente: 
$$P(y=k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

Diseña una función de pérdida para este modelo asumiendo que tenemos acceso a un conjunto de entrenamiento $\{x_i,y_i}\$ con I elementos 


2. Considere un problema de regresión multivariante en el que predecimos la altura de un individuo en metros y su peso en kilos a partir de unos datos x. Aquí, las unidades toman valores bastante diferentes. ¿Qué problemas cree que esto causa? Proponga dos soluciones.

3. El modelo de regresión logística utiliza una función lineal para predecir a cuál de dos clases $y \in \{0, 1\}$ pertenece una entrada x. Para una entrada unidimensional y una salida unidimensional, tiene dos parámetros, ϕ0 y ϕ1, y se define por:
$$P(y = 1|x) = sig(\phi_0 + \phi_1 x)$$

donde sig es la función sigmoide: $sig(z) = \frac{1}{1+e^{-z}}$

a. Grafica y vs x para este modelo con distintos valores de $\phi_0$ y $\phi_1$ y explica el valor cualitativo de cada parametro

b. ¿Cuál es una buena función de pérdida para este modelo? 

c. Calcula las parciales e la función de pérdida con respecto a los parámetros

d. Genera 10 puntos de una normal con media -1 y desviación estandard 1 y asignales $y=0$. Genera otros 10 puntos de una normal con media 1 y desviación estandard 1 y asignales $y=1$. Grafica estos puntos usando un heatmap en términos de los dos parámetros $\phi$

e. Es convexa esta función de pérdida? ¿Cómo pruebas esto? 


4. ¿Puede el descenso de gradiente (GD) (no estocástico) con una tasa de aprendizaje fija escapar de los mínimos locales?

5. Ejecutamos el algoritmo de SGD durante 1000 iteraciones en un conjunto de datos de tamaño 100 con un tamaño de lote de 20. ¿Para cuántas épocas estamos ejecutando el algoritmo?



### Ejercicios de Python: 

6. Regresión de Posion: usa la función de acá abajo para generar datos sintéticos parecidos a los de la pregunta 3. Implementa el modelo en Pytorch que prediga $s = f(x,\phi)$ y use $\lambda = e^s$. Entrena con la pérdida de Poisson. Compara entrenar con SGD, SGD con momentum y Adam.

a. Asegurate de entrenar con los mismos parámetros `lr=1e-2, epochs=80, batch_size=64` y `momentum = 0.9`, grafica las curvas de pérdida por época para los tres optimizadores. ¿Qué optimizador converge más rápido y por qué?

b. Ahora usa una tasa de aprendizaje distinta para cada algoritmo `lr_map = {'SGD': 1e-3, 'SGD_momentum': 1e-3, 'Adam': 1e-2}`. Grafica otra vez las curvas de pérdida. ¿Qué optimizador converge más rápido y por qué?

c. ¿Qué esta pasando cuando cambiamos las tasas de aprendizaje?


In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# Reproducibilidad: fija la semilla
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)
device = torch.device("cpu")  # o "cuda" si tienes GPU


def generate_p1_data(N=2000):
    """
    Datos para P1 (Poisson counts):
     - features: [hour_sin, hour_cos, lat, lon, neigh_onehot(3)]
     - latent s = linear(X)*w + small nonlinear rush-hour term
     - lambda = exp(s), y ~ Poisson(lambda)
    """
    rng = np.random.RandomState(0)
    hours = rng.randint(0, 24, size=(N, 1)).astype(np.float32)
    hour_rad = 2 * math.pi * hours / 24.0
    hour_sin = np.sin(hour_rad).astype(np.float32)
    hour_cos = np.cos(hour_rad).astype(np.float32)
    lat = (40.0 + rng.normal(scale=0.01, size=(N,1))).astype(np.float32)
    lon = (-74.0 + rng.normal(scale=0.01, size=(N,1))).astype(np.float32)
    neigh = rng.randint(0, 3, size=(N,))
    neigh_onehot = np.zeros((N,3), dtype=np.float32)
    neigh_onehot[np.arange(N), neigh] = 1.0

    X = np.concatenate([hour_sin, hour_cos, lat, lon, neigh_onehot], axis=1)
    # pesos "verdaderos" escogidos para producir lambdas razonables
    w = np.array([0.8, -0.6, 30.0, -20.0, 0.5, 0.2, -0.1], dtype=np.float32)
    s_linear = X @ w.reshape(-1,1)
    # horario pico (mañana y tarde)
    rush = 1.0 * np.exp(-((hours - 8)**2)/8.0) + 1.2 * np.exp(-((hours - 18)**2)/10.0)
    s = s_linear + 0.3 * rush
    lam = np.clip(np.exp(s), 1e-6, 50.0)
    y = rng.poisson(lam).astype(np.float32)

    return torch.from_numpy(X).float(), torch.from_numpy(y).float(), {'w_true': w}

P1: X1.shape = torch.Size([2000, 7]) y1.shape = torch.Size([2000, 1])
P2: X2.shape = torch.Size([2000, 5]) Y2.shape = torch.Size([2000, 2])


/var/folders/bl/8nlz54wj6vz1h95x_pcnxjc40000gn/T/ipykernel_22386/1880895205.py:48: RuntimeWarning: overflow encountered in exp
  lam = np.clip(np.exp(s), 1e-6, 50.0)


7. Generamos un dataset sintético con dos salidas: altura (metros) y peso (kilogramos). Entrena y compara dos versiones de la misma red neuronal:
- Modelo A: entrena directamente sobre targets sin normalizar.
- Modelo B: normaliza los targets (z-score) durante el entrenamiento; al evaluar des-normaliza las predicciones para reportar métricas en las unidades reales.

a. Grafica la función de pérdida (MSE) global (train y val) en unidades reales por época para ambos modelos en la misma figura

b. Grafica RMSE por salida (altura, peso) en validación por época para ambos modelos 

c. Tabla con RMSE final en validación (altura y peso) para ambos modelos

d. Explica por qué la normalización de los targets ayuda 

c. ¿Qué harias si quisieras priorizar la precisión en altura sobre el peso? (pista: piensa en la función de pérdia o normaliza de forma distinta)

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

# -----------------------------
# Reproducibilidad: fija la semilla
# -----------------------------
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)
device = torch.device("cpu")  # o "cuda" si tienes GPU

def generate_p2_data(N=2000):
    """
    Datos para P2 (height, weight):
      - X: 5 features N(0,1)
      - Height ~ 1.7 + small linear effect + noise(std ~0.08)
      - Weight ~ 70 + larger linear effect + noise(std ~7)
    """
    rng = np.random.RandomState(1)
    X = rng.normal(size=(N,5)).astype(np.float32)
    w_h = np.array([0.02, -0.01, 0.03, 0.0, 0.01], dtype=np.float32)
    w_w = np.array([1.2, -0.8, 0.5, 0.3, 0.1], dtype=np.float32)
    height = 1.7 + X @ w_h.reshape(-1,1) + rng.normal(scale=0.08, size=(N,1)).astype(np.float32)
    weight = 70.0 + X @ w_w.reshape(-1,1) + rng.normal(scale=7.0, size=(N,1)).astype(np.float32)
    Y = np.concatenate([height, weight], axis=1).astype(np.float32)
    return torch.from_numpy(X).float(), torch.from_numpy(Y).float(), {'w_h': w_h, 'w_w': w_w}

X2, Y2, info2 = generate_p2_data(N=2000)   # P2
print("P2: X2.shape =", X2.shape, "Y2.shape =", Y2.shape)

P1: X1.shape = torch.Size([2000, 7]) y1.shape = torch.Size([2000, 1])
P2: X2.shape = torch.Size([2000, 5]) Y2.shape = torch.Size([2000, 2])


/var/folders/bl/8nlz54wj6vz1h95x_pcnxjc40000gn/T/ipykernel_22386/1880895205.py:48: RuntimeWarning: overflow encountered in exp
  lam = np.clip(np.exp(s), 1e-6, 50.0)
